# Agente de IA com múltiplas ferramentas

## Prática no Google Colab

Nesta atividade, vamos construir um agente que recebe perguntas em linguagem natural e escolhe qual ferramenta Python deve utilizar para responder.

A ideia principal é observar que o agente não precisa ter uma única função. Ele pode escolher entre diferentes ferramentas de acordo com o pedido do usuário.

**Não vamos utilizar SQLite nem arquivos externos nesta prática.** O foco é entender o funcionamento de um agente e a seleção de ferramentas (Tool Calling).

Ao final, o agente poderá:
- calcular o custo de um consumo de energia;
- converter potência de watts para quilowatts;
- estimar o consumo de um equipamento;
- calcular a economia após uma redução no consumo.

## 1. Preparar o ambiente

Vamos instalar o Ollama no ambiente temporário do Google Colab. Assim, não é necessário instalar o programa no computador do laboratório.

A instalação e o download do modelo dependem de conexão com a internet e dos recursos disponíveis na sessão do Colab.

In [1]:
!sudo apt update
!sudo apt install -y pciutils
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [2,000 kB]
Get:6 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Hit:7 http://archive.ubuntu.com/ubuntu noble InRelease
Get:8 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [78.0 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:10 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.5 MB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-dri

In [26]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)
print("Servidor Ollama iniciado.")

Servidor Ollama iniciado.


In [27]:
!ollama pull qwen3:1.7b

In [4]:
!pip -q install ollama

In [28]:
import ollama
import json

MODELO = "qwen3:1.7b"

print("Ambiente preparado.")

Ambiente preparado.


## 2. Criar as ferramentas em Python

Um agente de IA pode utilizar funções externas para realizar tarefas específicas. Essas funções são chamadas de ferramentas (tools).

Nesta atividade, vamos criar quatro ferramentas relacionadas aos conceitos básicos de Engenharia Elétrica que já estudamos em aula:

1. `calcular_corrente`: calcula a corrente elétrica em um circuito utilizando a Lei de Ohm.
2. `calcular_tensao`: calcula a tensão elétrica a partir da corrente e da resistência.
3. `calcular_potencia`: calcula a potência elétrica de um circuito a partir da tensão e da corrente.
4. `calcular_resistencia`: calcula a resistência elétrica utilizando a tensão e a corrente.

Cada ferramenta será implementada como uma função em Python, recebendo os valores necessários e retornando o resultado do cálculo.

Primeiro, vamos criar e testar cada função individualmente, sem utilizar inteligência artificial. Assim, garantimos que as ferramentas estejam funcionando corretamente antes de integrá-las ao agente de IA.

In [36]:
def calcular_corrente(tensao, resistencia):
    """
    Calcula a corrente elétrica pela Lei de Ohm.
    I = V / R
    """
    corrente = tensao / resistencia
    return round(corrente, 3)


def calcular_tensao(corrente, resistencia):
    """
    Calcula a tensão elétrica pela Lei de Ohm.
    V = R * I
    """
    tensao = resistencia * corrente
    return round(tensao, 3)


def calcular_potencia(tensao, corrente):
    """
    Calcula a potência elétrica.
    P = V * I
    """
    potencia = tensao * corrente
    return round(potencia, 3)


def calcular_resistencia(tensao, corrente):
    """
    Calcula a resistência elétrica pela Lei de Ohm.
    R = V / I
    """
    resistencia = tensao / corrente
    return round(resistencia, 3)


# Testando as ferramentas diretamente

print("Corrente:", calcular_corrente(12, 4), "A")
print("Tensão:", calcular_tensao(2, 10), "V")
print("Potência:", calcular_potencia(12, 2), "W")
print("Resistência:", calcular_resistencia(12, 2), "Ω")

Corrente: 3.0 A
Tensão: 20 V
Potência: 24 W
Resistência: 6.0 Ω


## 3. Apresentar as ferramentas ao modelo

O modelo precisa saber quais ferramentas existem, para que servem e quais informações cada uma recebe.

Vamos fornecer essas descrições em uma instrução. Quando receber uma pergunta, o LLM deverá escolher **uma ferramenta** e retornar seus argumentos em JSON.

In [38]:
DESCRICAO_FERRAMENTAS = '''
Você é o componente de decisão de um agente de Inteligência Artificial
que auxilia na resolução de problemas básicos de Engenharia Elétrica.

Sua tarefa é interpretar a pergunta do usuário e selecionar a ferramenta
Python mais adequada para realizar o cálculo solicitado.

FERRAMENTAS DISPONÍVEIS:

1. calcular_corrente
   Calcula a corrente elétrica de um circuito utilizando a Lei de Ohm.
   Fórmula: I = V / R

   Argumentos:
   - tensao: tensão elétrica em volts (V).
   - resistencia: resistência elétrica em ohms (Ω).

   Utilize quando o usuário quiser descobrir a corrente elétrica
   e informar os valores de tensão e resistência.

2. calcular_tensao
   Calcula a tensão elétrica utilizando a Lei de Ohm.
   Fórmula: V = R * I

   Argumentos:
   - corrente: corrente elétrica em amperes (A).
   - resistencia: resistência elétrica em ohms (Ω).

   Utilize quando o usuário quiser descobrir a tensão elétrica
   e informar os valores de corrente e resistência.

3. calcular_potencia
   Calcula a potência elétrica de um circuito.
   Fórmula: P = V * I

   Argumentos:
   - tensao: tensão elétrica em volts (V).
   - corrente: corrente elétrica em amperes (A).

   Utilize quando o usuário quiser descobrir a potência elétrica
   e informar os valores de tensão e corrente.

4. calcular_resistencia
   Calcula a resistência elétrica utilizando a Lei de Ohm.
   Fórmula: R = V / I

   Argumentos:
   - tensao: tensão elétrica em volts (V).
   - corrente: corrente elétrica em amperes (A).

   Utilize quando o usuário quiser descobrir a resistência elétrica
   e informar os valores de tensão e corrente.

REGRAS PARA ESCOLHER A FERRAMENTA:

1. Escolha somente uma ferramenta por solicitação.
2. Identifique qual grandeza elétrica o usuário deseja descobrir.
3. Utilize os valores informados pelo usuário, respeitando suas unidades.
4. Não invente valores nem complete informações que não foram fornecidas.
5. Não confunda as grandezas elétricas. Por exemplo, tensão não é
   corrente, e potência não é resistência.
6. Não realize os cálculos. A ferramenta Python será responsável
   por executar a fórmula.
7. Se faltar algum argumento necessário, não escolha uma ferramenta.
   Informe que faltam dados para realizar o cálculo.

FORMATO DA RESPOSTA:

Retorne SOMENTE um JSON válido, sem explicações, comentários ou markdown.

Quando todos os argumentos estiverem disponíveis, utilize exatamente
esta estrutura:

{
    "ferramenta": "nome_da_ferramenta",
    "argumentos": {
        "nome_do_argumento": 123
    }
}

Se faltarem informações para realizar o cálculo, retorne:

{
    "ferramenta": null,
    "argumentos": {},
    "mensagem": "Informe quais dados estão faltando."
}

Não adicione argumentos que não sejam exigidos pela ferramenta.
Não inclua unidades nos valores numéricos.
'''

print(DESCRICAO_FERRAMENTAS)


Você é o componente de decisão de um agente de Inteligência Artificial
que auxilia na resolução de problemas básicos de Engenharia Elétrica.

Sua tarefa é interpretar a pergunta do usuário e selecionar a ferramenta
Python mais adequada para realizar o cálculo solicitado.

FERRAMENTAS DISPONÍVEIS:

1. calcular_corrente
   Calcula a corrente elétrica de um circuito utilizando a Lei de Ohm.
   Fórmula: I = V / R

   Argumentos:
   - tensao: tensão elétrica em volts (V).
   - resistencia: resistência elétrica em ohms (Ω).

   Utilize quando o usuário quiser descobrir a corrente elétrica
   e informar os valores de tensão e resistência.

2. calcular_tensao
   Calcula a tensão elétrica utilizando a Lei de Ohm.
   Fórmula: V = R * I

   Argumentos:
   - corrente: corrente elétrica em amperes (A).
   - resistencia: resistência elétrica em ohms (Ω).

   Utilize quando o usuário quiser descobrir a tensão elétrica
   e informar os valores de corrente e resistência.

3. calcular_potencia
   Calcu

## 4. Criar a etapa de decisão do agente

A função abaixo envia a pergunta ao modelo. O modelo interpreta o pedido e escolhe qual ferramenta deve ser utilizada.

Por enquanto, vamos apenas observar a decisão, sem executar a ferramenta.

In [39]:
def extrair_json(texto):
    """
    Remove possíveis marcações de Markdown e converte
    a resposta do modelo para um dicionário Python.
    """
    texto = texto.replace("```json", "").replace("```", "").strip()
    return json.loads(texto)


def escolher_ferramenta(pergunta):
    """
    Envia a pergunta ao modelo de IA e recebe sua decisão
    sobre qual ferramenta Python deve ser utilizada.
    """

    resposta = ollama.chat(
        model=MODELO,
        messages=[
            {"role": "system", "content": DESCRICAO_FERRAMENTAS},
            {"role": "user", "content": pergunta}
        ],
        options={"temperature": 0}
    )

    # Converte a resposta do modelo para um dicionário Python
    decisao = extrair_json(resposta["message"]["content"])

    # Lista de ferramentas que o agente pode utilizar
    permitidas = {
        "calcular_corrente",
        "calcular_tensao",
        "calcular_potencia",
        "calcular_resistencia"
    }

    ferramenta = decisao.get("ferramenta")
    argumentos = decisao.get("argumentos")

    # Verifica se o modelo identificou corretamente a ferramenta
    if ferramenta is None:
        return decisao

    if ferramenta not in permitidas:
        raise ValueError("O modelo escolheu uma ferramenta não permitida.")

    # Verifica se os argumentos estão no formato esperado
    if not isinstance(argumentos, dict):
        raise ValueError("Os argumentos não estão no formato esperado.")

    return decisao


# Testando a seleção de ferramenta
pergunta = "Qual é a corrente elétrica em um resistor de 4 ohms ligado a uma tensão de 12 volts?"

decisao = escolher_ferramenta(pergunta)

print("Pergunta:", pergunta)
print("\nDecisão do agente:")
print(json.dumps(decisao, ensure_ascii=False, indent=2))

Pergunta: Qual é a corrente elétrica em um resistor de 4 ohms ligado a uma tensão de 12 volts?

Decisão do agente:
{
  "ferramenta": "calcular_corrente",
  "argumentos": {
    "tensao": 12,
    "resistencia": 4
  }
}


## 5. Criar o executor de ferramentas

Agora vamos criar uma função que recebe a decisão do modelo e executa a ferramenta escolhida.

Observe que o LLM não realiza diretamente o cálculo. Ele seleciona a função e informa os argumentos. Quem executa a operação é o Python.

In [41]:
def executar_ferramenta(decisao):
    """
    Executa a ferramenta Python selecionada pelo agente,
    utilizando os argumentos fornecidos pelo modelo.
    """

    nome = decisao["ferramenta"]
    args = decisao["argumentos"]

    # Calcula a corrente elétrica: I = V / R
    if nome == "calcular_corrente":

        tensao = float(args["tensao"])
        resistencia = float(args["resistencia"])

        if resistencia == 0:
            raise ValueError("A resistência não pode ser zero.")

        return calcular_corrente(tensao, resistencia)

    # Calcula a tensão elétrica: V = R * I
    elif nome == "calcular_tensao":

        corrente = float(args["corrente"])
        resistencia = float(args["resistencia"])

        return calcular_tensao(corrente, resistencia)

    # Calcula a potência elétrica: P = V * I
    elif nome == "calcular_potencia":

        tensao = float(args["tensao"])
        corrente = float(args["corrente"])

        return calcular_potencia(tensao, corrente)

    # Calcula a resistência elétrica: R = V / I
    elif nome == "calcular_resistencia":

        tensao = float(args["tensao"])
        corrente = float(args["corrente"])

        if corrente == 0:
            raise ValueError("A corrente não pode ser zero.")

        return calcular_resistencia(tensao, corrente)

    else:
        raise ValueError("Ferramenta não autorizada.")


# Executando a ferramenta selecionada pelo agente
resultado = executar_ferramenta(decisao)

print("Resultado da ferramenta:", resultado)

Resultado da ferramenta: 3.0


## 6. Gerar uma resposta para o usuário

Depois da execução da ferramenta, vamos enviar o resultado ao modelo para que ele explique o que foi calculado.

O valor numérico vem da função Python. O modelo apenas organiza a explicação em linguagem natural.

In [42]:
def formular_resposta(pergunta, decisao, resultado):

    resposta = ollama.chat(
        model=MODELO,
        messages=[
            {
                "role": "system",
                "content": """
                Você é um assistente de Engenharia Elétrica.

                Responda em português, utilizando uma linguagem simples,
                clara e adequada para estudantes iniciantes.

                REGRAS:
                1. Explique o resultado do cálculo realizado.
                2. Apresente a fórmula correspondente à ferramenta utilizada.
                3. Explique brevemente o significado das grandezas envolvidas.
                4. Informe o resultado com sua unidade de medida correta.
                5. Utilize exclusivamente os valores e o resultado fornecidos
                   pelo Python.
                6. Não invente valores nem realize novos cálculos.
                7. Não altere o resultado retornado pela ferramenta.
                8. Seja objetivo e evite explicações desnecessárias.
                """
            },
            {
                "role": "user",
                "content": (
                    f"Pergunta do usuário: {pergunta}\n"
                    f"Ferramenta utilizada: {decisao['ferramenta']}\n"
                    f"Argumentos utilizados: {decisao['argumentos']}\n"
                    f"Resultado calculado pelo Python: {resultado}\n\n"
                    "Elabore uma resposta explicando o resultado."
                )
            }
        ],
        options={"temperature": 0}
    )

    return resposta["message"]["content"]


# Gerando a resposta final do agente
texto_resposta = formular_resposta(pergunta, decisao, resultado)

print("Resposta do agente:")
print(texto_resposta)

Resposta do agente:
A corrente elétrica em um resistor é calculada usando a lei de Ohm, que relaciona tensão, resistência e corrente pela fórmula:  
**I = V/R**  
Onde:  
- **I** é a corrente (em ampères, A),  
- **V** é a tensão (em volts, V),  
- **R** é a resistência (em ohms, Ω).  

Aplicando os valores fornecidos:  
**Tensão (V) = 12 V**  
**Resistência (R) = 4 Ω**  

Calculando:  
**I = 12 V / 4 Ω = 3 A**  

Portanto, a corrente elétrica é **3 ampères**.  
O resultado é obtido dividindo a tensão pela resistência, o que é o que a fórmula diz.


## 7. Reunir as etapas em um único agente

Agora vamos juntar tudo em uma função. O usuário faz uma pergunta e o agente:

1. interpreta a solicitação;
2. escolhe uma ferramenta;
3. informa os argumentos;
4. executa a função Python;
5. apresenta uma resposta.

In [43]:
def agente(pergunta):

    print("=" * 60)
    print("PERGUNTA:", pergunta)
    print("=" * 60)

    # ETAPA 1: Interpretar a pergunta e selecionar a ferramenta
    print("\n[1] INTERPRETAÇÃO E SELEÇÃO")

    decisao = escolher_ferramenta(pergunta)

    if decisao["ferramenta"] is None:
        print("Não foi possível realizar o cálculo.")
        print(decisao.get("mensagem", "Faltam informações."))
        return None

    print("Ferramenta selecionada:", decisao["ferramenta"])
    print("Argumentos:", decisao["argumentos"])

    # ETAPA 2: Executar a ferramenta Python
    print("\n[2] EXECUÇÃO")

    resultado = executar_ferramenta(decisao)

    print("Resultado do cálculo:", resultado)

    # ETAPA 3: Gerar a resposta em linguagem natural
    print("\n[3] RESPOSTA DO AGENTE")

    resposta = formular_resposta(
        pergunta,
        decisao,
        resultado
    )

    print(resposta)
    print("\n" + "=" * 60)

    return {
        "decisao": decisao,
        "resultado": resultado,
        "resposta": resposta
    }


# Testando o agente completo
agente(
    "Qual é a corrente elétrica em um resistor de 4 ohms "
    "ligado a uma fonte de 12 volts?"
)

PERGUNTA: Qual é a corrente elétrica em um resistor de 4 ohms ligado a uma fonte de 12 volts?

[1] INTERPRETAÇÃO E SELEÇÃO
Ferramenta selecionada: calcular_corrente
Argumentos: {'tensao': 12, 'resistencia': 4}

[2] EXECUÇÃO
Resultado do cálculo: 3.0

[3] RESPOSTA DO AGENTE
**Resultado:** A corrente elétrica é de 3 ampères (A).  

**Fórmula utilizada:**  
$ I = \frac{V}{R} $  
Onde:  
- $ I $ = Corrente elétrica (em ampères, A)  
- $ V $ = Tensão elétrica (em volts, V)  
- $ R $ = Resistência elétrica (em ohms, Ω)  

**Explicação:**  
A fórmula de Ohm ( $ I = \frac{V}{R} $ ) mostra que a corrente é diretamente proporcional à tensão e inversamente proporcional à resistência. Com uma tensão de 12 V e uma resistência de 4 Ω, a corrente calculada é $ \frac{12}{4} = 3 $ A.  

**Unidade:** A (ampère) é a unidade padrão para corrente elétrica.



{'decisao': {'ferramenta': 'calcular_corrente',
  'argumentos': {'tensao': 12, 'resistencia': 4}},
 'resultado': 3.0,
 'resposta': '**Resultado:** A corrente elétrica é de 3 ampères (A).  \n\n**Fórmula utilizada:**  \n$ I = \\frac{V}{R} $  \nOnde:  \n- $ I $ = Corrente elétrica (em ampères, A)  \n- $ V $ = Tensão elétrica (em volts, V)  \n- $ R $ = Resistência elétrica (em ohms, Ω)  \n\n**Explicação:**  \nA fórmula de Ohm ( $ I = \\frac{V}{R} $ ) mostra que a corrente é diretamente proporcional à tensão e inversamente proporcional à resistência. Com uma tensão de 12 V e uma resistência de 4 Ω, a corrente calculada é $ \\frac{12}{4} = 3 $ A.  \n\n**Unidade:** A (ampère) é a unidade padrão para corrente elétrica.'}

## 8. Testar diferentes ferramentas

Execute as perguntas abaixo e observe como o agente escolhe ferramentas diferentes conforme o pedido.

Atenção: modelos pequenos podem interpretar alguns pedidos incorretamente. Se isso acontecer, tente reformular a pergunta ou confira os argumentos escolhidos.

In [44]:
perguntas = [
    "Qual é a corrente elétrica em um resistor de 4 ohms ligado a uma fonte de 12 volts?",

    "Um circuito possui corrente de 2 amperes e resistência de 10 ohms. Qual é a tensão?",

    "Um equipamento funciona com tensão de 220 volts e corrente de 5 amperes. Qual é sua potência?",

    "Um circuito apresenta tensão de 24 volts e corrente de 3 amperes. Qual é sua resistência?"
]


# Executando o agente para cada pergunta
for i, pergunta_teste in enumerate(perguntas, start=1):

    print(f"\n{'=' * 60}")
    print(f"TESTE {i}")
    print("=" * 60)

    agente(pergunta_teste)


TESTE 1
PERGUNTA: Qual é a corrente elétrica em um resistor de 4 ohms ligado a uma fonte de 12 volts?

[1] INTERPRETAÇÃO E SELEÇÃO
Ferramenta selecionada: calcular_corrente
Argumentos: {'tensao': 12, 'resistencia': 4}

[2] EXECUÇÃO
Resultado do cálculo: 3.0

[3] RESPOSTA DO AGENTE
**Resultado do cálculo:**  
A corrente elétrica calculada é **3 ampères (A)**.  

**Fórmula utilizada:**  
A lei de Ohm, que relaciona corrente, tensão e resistência:  
$$
I = \frac{V}{R}
$$  
Onde:  
- $ I $ = Corrente elétrica (em ampères, A)  
- $ V $ = Tensão elétrica (em volts, V)  
- $ R $ = Resistência elétrica (em ohms, Ω)  

**Explicação das grandezas:**  
- A **tensão** é a força que impulsiona a corrente no circuito (12 V).  
- A **resistência** é a oposta ao fluxo de corrente (4 Ω).  
- A **corrente** é o fluxo de elétrons que passa pelo circuito (3 A).  

**Resultado final:**  
A corrente elétrica no resistor é **3 A**.


TESTE 2
PERGUNTA: Um circuito possui corrente de 2 amperes e resistência d

## 9. O que você deve observar

Ao final, você deverá conseguir explicar:

- Qual é o papel do modelo de linguagem?
- Como o agente escolhe entre diferentes ferramentas?
- O que são os argumentos de uma ferramenta?
- Qual parte do processo é executada pelo Python?
- Por que o modelo não deve executar diretamente qualquer código que ele sugerir?

**Ideia principal:** o agente utiliza o LLM para interpretar e decidir, mas utiliza ferramentas externas para executar tarefas concretas.